In [1]:
suppressMessages({
    require(Seurat)
    require(dplyr)
    require(igraph)
    require(ggpubr)
})

In [2]:
# load orthogroups
orthogroups <- read.delim('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/02.gene_relationships/run4/results/Ortho_pipeline/OrthoFinder/Orthogroups/Orthogroups.tsv')
# at least one copy for 4 species
orthogroups <- orthogroups %>% select(c('Orthogroup', 'Pmar', 'Pvit', 'Mmus', 'Hsap'))  %>% 
    filter(Pmar != '' | Pvit != '' | Mmus != '' | Hsap != '')

In [3]:
# get TFs for each species
Hsap_TFs <- read.table('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/07.species_signals/6.TF_vs_species_signals/TFs/Hsap_TFs.txt', header = F)$V1
Mmus_TFs <- read.table('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/07.species_signals/6.TF_vs_species_signals/TFs/Mmus_TFs.name.txt', header = F)$V1
Pvit_TFs <- read.table('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/07.species_signals/6.TF_vs_species_signals/TFs/Pvit.predicted_TFs.txt', header = T)
Pmar_TFs <- read.table('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/07.species_signals/6.TF_vs_species_signals/TFs/Pmar.predicted_TFs.txt', header = T)
Pvit_TFs <- Pvit_TFs[Pvit_TFs$prediction == 'True', 1]
Pmar_TFs <- Pmar_TFs[Pmar_TFs$prediction == 'True', 1]

In [4]:
# get ohnologues and SSD paralogues info
oh_pa_family <- readRDS('Combined.SSD_WGD.pairs.rds')
paralog_gene_type <- readRDS('gene_type.rds')

In [5]:
# gene annotation for gene ID and gene name
Hsap_ID <- read.delim('0.bin/Hsap.info', header = T)
Hsap_ID <- Hsap_ID[Hsap_ID$Gene.type == 'protein_coding', ]
Hsap_ID[Hsap_ID$Gene.name == '', 'Gene.name'] <- Hsap_ID[Hsap_ID$Gene.name == '', 'Gene.stable.ID']

Mmus_ID <- read.delim('0.bin/Mmus.info', header = T)
Mmus_ID <- Mmus_ID[Mmus_ID$Gene.type == 'protein_coding', ]
Mmus_ID[Mmus_ID$Gene.name == '', 'Gene.name'] <- Mmus_ID[Mmus_ID$Gene.name == '', 'Gene.stable.ID']

In [6]:
Hsap <- readRDS('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/02.atlas_final/2.samap/4.final/Hsap.non_neurons.iter_cluster_annotated.rds')
Mmus <- readRDS('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/02.atlas_final/2.samap/4.final/Mmus.non_neurons.iter_cluster_annotated.rds')
Pvit <- readRDS('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/02.atlas_final/2.samap/4.final/Pvit.non_neurons.iter_cluster_annotated.rds')

In [7]:
# retain  astrocytes and oligodendrocyte lineages
Idents(Hsap) <- 'Refined family'
Idents(Mmus) <- 'Refined family'
Idents(Pvit) <- 'Refined family'
Hsap <- subset(Hsap, idents = c('Astrocytes', 'Oligodendrocytes'))
Mmus <- subset(Mmus, idents = c('Astrocytes', 'Oligodendrocytes'))
Pvit <- subset(Pvit, idents = c('Astrocytes', 'Oligodendrocytes'))

In [8]:
# subset to same number of cells
subset_further <- function(obj, number){
    sampled_df <- obj@meta.data
    sampled_df$cellname <- rownames(sampled_df)
    sampled_df <- sampled_df %>% group_by(`Refined family`) %>% slice_sample(n = number) %>% ungroup()
    obj <- subset(obj, cells = as.character(sampled_df$cellname))
    return(obj)
}

Hsap <- subset_further(Hsap, 5000)
Mmus <- subset_further(Mmus, 5000)
Pvit <- subset_further(Pvit, 5000)

In [9]:
# Find markers between AST and Oligo
get_markers <- function(object, label){
    markers <- FindAllMarkers(object, only.pos = T, verbose = F)
    markers <- markers %>% filter(p_val_adj < 0.01 & avg_log2FC >= 0.58 & pct.1 > 0.1)
    # rank by pct and avg log2FC
    markers <- markers %>% group_by(cluster) %>% 
        arrange(
            desc(pct.1 / (pct.1 + pct.2)),
            desc(avg_log2FC),
            .by_group = TRUE
        ) %>% ungroup() %>% as.data.frame()
    markers$species <- label
    return(markers)
}

In [10]:
# find markers and add gene names or IDs
Idents(Hsap) <- 'Refined family'
Idents(Mmus) <- 'Refined family'
Idents(Pvit) <- 'Refined family'
Hsap_markers <- get_markers(Hsap, 'Hsap')
Mmus_markers <- get_markers(Mmus, 'Mmus')
Pvit_markers <- get_markers(Pvit, 'Pvit')

In [11]:
# add gene name, TFs, ohnologs vs SSD paralogs
# add gene categories
add_type <- function(markers, label){
    res <- vapply(markers$gene, FUN = function(g){
        tmp <- paralog_gene_type[[label]]
        if (g %in% tmp$gene){
            res <- tmp[tmp$gene == g, 'type']
        } else {
            res <- 'Others'
        }
    }, FUN.VALUE = character(1))
    markers$type <- res
    return(markers)
}

Hsap_markers$gene_name <- Hsap_ID[match(Hsap_markers$gene, Hsap_ID$Gene.stable.ID), 'Gene.name']
Hsap_markers$TF <- Hsap_markers$gene %in% Hsap_TFs
Hsap_markers <- add_type(Hsap_markers, 'Hsap')

Mmus_markers$gene_name <- Mmus_ID[match(Mmus_markers$gene, Mmus_ID$Gene.name), 'Gene.stable.ID']
Mmus_markers$TF <- Mmus_markers$gene %in% Mmus_TFs
Mmus_markers <- add_type(Mmus_markers, 'Mmus')

Pvit_markers$gene_name <- Pvit_markers$gene
Pvit_markers$TF <- Pvit_markers$gene %in% Pvit_TFs
Pvit_markers <- add_type(Pvit_markers, 'Pvit')


In [12]:
# add families information, get family level ohnologs and SSD paralogues
get_family <- function(genes, label){
    # get family in lists
    g <- graph_from_data_frame(oh_pa_family[[label]][,1:2], directed = FALSE)
    components <- components(g)$membership
    family <- split(names(components), components)
    
    # Find the matching family for the gene
    results <- vapply(genes, FUN = function(gene) {
        res <- lapply(family, function(x) {
            if (gene %in% x) return(x)
            })
        # Filter out NULL values (elements where the gene wasn't found)
        res <- Filter(Negate(is.null), res)
        
        # If a match is found, return the first result; otherwise, return an empty string
        if (length(res) > 0) {
            return(paste0(unlist(res), collapse = ","))
        } else {
            return("")
        }
    }, FUN.VALUE = character(1))
    return(results)
}

In [13]:
Hsap_markers$family <- get_family(Hsap_markers$gene, 'Hsap')
Mmus_markers$family <- get_family(Mmus_markers$gene, 'Mmus')
Pvit_markers$family <- get_family(Pvit_markers$gene, 'Pvit')

In [14]:
# add orthogroup information
get_orthogroup <- function(markers, species){
    tmp = orthogroups[, c('Orthogroup', species)] %>% filter(species != '') %>% 
        tidyr::separate_rows(species, sep = ", ") %>% as.data.frame()
    tmp <- tmp[match(markers, tmp[[species]]),1]
    return(tmp)
}

Hsap_markers$orthogroup <- get_orthogroup(Hsap_markers$gene, 'Hsap')
Mmus_markers$orthogroup <- get_orthogroup(Mmus_markers$gene, 'Mmus')
Pvit_markers$orthogroup <- get_orthogroup(Pvit_markers$gene, 'Pvit')

Warning message:
“Using an external vector in selections was deprecated in tidyselect 1.1.0.
ℹ Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(species)

  # Now:
  data %>% select(all_of(species))

See <https://tidyselect.r-lib.org/reference/faq-external-vector.html>.”


In [15]:
# function to get WGD paralogue family with different members used in AST and Oligo OR 
# WGD paralogue family only used in one of above two cell types.

get_markers_for_divergence_WGD <- function(markers, label){
    markers <- markers %>% filter(type == 'WGD')
    tmp1 <- unique(unlist(markers %>% filter(cluster == 'Astrocytes') %>% select(family)))
    tmp2 <- unique(unlist(markers %>% filter(cluster == 'Oligodendrocytes') %>% select(family)))
    tmp1 <- tmp1[tmp1 != '']
    tmp2 <- tmp2[tmp2 != '']
    
    cat(paste0(label, ':\nNumber of WGD paralogue family involved:', length(unique(c(tmp1,tmp2))),
               ';\nNumber of WGD paralogue family involved only in one of AST and Oligo:', sum(table(c(tmp1,tmp2)) == 1),
               ';\nNumber of WGD paralogue family involved in these two:', sum(table(c(tmp1,tmp2)) == 2), '\n'))
    
    x <- (table(c(tmp1,tmp2)) == 2)
    interested_1 <- names(x)[x]
    markers_1 <- markers %>% filter(family %in% interested_1)
    
    x <- (table(c(tmp1,tmp2)) == 1)
    interested_2 <- names(x)[x]
    markers_2 <- markers %>% filter(family %in% interested_2)
    return(list(a = markers_1, b = markers_2))
}

# function to get SSD paralogue family with different members used in AST and Oligo
get_markers_for_divergence_SSD <- function(markers, label){
    markers <- markers %>% filter(type == 'SSD')
    tmp1 <- unique(unlist(markers %>% filter(cluster == 'Astrocytes') %>% select(family)))
    tmp2 <- unique(unlist(markers %>% filter(cluster == 'Oligodendrocytes') %>% select(family)))
    tmp1 <- tmp1[tmp1 != '']
    tmp2 <- tmp2[tmp2 != '']
    
    cat(paste0(label, ':\nNumber of SSD paralogue family involved:', length(unique(c(tmp1,tmp2))),
               ';\nNumber of SSD paralogue family involved only in one of AST and Oligo:', sum(table(c(tmp1,tmp2)) == 1),
               ';\nNumber of SSD paralogue family involved in these two:', sum(table(c(tmp1,tmp2)) == 2), '\n'))
    
    x <- (table(c(tmp1,tmp2)) == 2)
    interested_1 <- names(x)[x]
    markers_1 <- markers %>% filter(family %in% interested_1)
    
    x <- (table(c(tmp1,tmp2)) == 1)
    interested_2 <- names(x)[x]
    markers_2 <- markers %>% filter(family %in% interested_2)
    return(list(a = markers_1, b = markers_2))
}

In [16]:
Hsap_interesting_WGD <- get_markers_for_divergence_WGD(Hsap_markers, 'Hsap')
Hsap_interesting_SSD <- get_markers_for_divergence_SSD(Hsap_markers, 'Hsap')
Mmus_interesting_WGD <- get_markers_for_divergence_WGD(Mmus_markers, 'Mmus')
Mmus_interesting_SSD <- get_markers_for_divergence_SSD(Mmus_markers, 'Mmus')
Pvit_interesting_WGD <- get_markers_for_divergence_WGD(Pvit_markers, 'Pvit')
Pvit_interesting_SSD <- get_markers_for_divergence_SSD(Pvit_markers, 'Pvit')

Hsap:
Number of WGD paralogue family involved:746;
Number of WGD paralogue family involved only in one of AST and Oligo:642;
Number of WGD paralogue family involved in these two:104
Hsap:
Number of SSD paralogue family involved:576;
Number of SSD paralogue family involved only in one of AST and Oligo:512;
Number of SSD paralogue family involved in these two:64
Mmus:
Number of WGD paralogue family involved:424;
Number of WGD paralogue family involved only in one of AST and Oligo:395;
Number of WGD paralogue family involved in these two:29
Mmus:
Number of SSD paralogue family involved:348;
Number of SSD paralogue family involved only in one of AST and Oligo:326;
Number of SSD paralogue family involved in these two:22
Pvit:
Number of WGD paralogue family involved:434;
Number of WGD paralogue family involved only in one of AST and Oligo:408;
Number of WGD paralogue family involved in these two:26
Pvit:
Number of SSD paralogue family involved:226;
Number of SSD paralogue family involved onl

In [17]:
# AST conserved marker family, Oligo conserved marker family
tmp1 = c(
    unique(unlist(Hsap_markers %>% filter(cluster == 'Astrocytes' & TF) %>% select('orthogroup'))),
    unique(unlist(Mmus_markers %>% filter(cluster == 'Astrocytes' & TF) %>% select('orthogroup'))),
    unique(unlist(Pvit_markers %>% filter(cluster == 'Astrocytes' & TF) %>% select('orthogroup')))
)

tmp2 = c(
    unique(unlist(Hsap_markers %>% filter(cluster == 'Oligodendrocytes' & TF) %>% select('orthogroup'))),
    unique(unlist(Mmus_markers %>% filter(cluster == 'Oligodendrocytes' & TF) %>% select('orthogroup'))),
    unique(unlist(Pvit_markers %>% filter(cluster == 'Oligodendrocytes' & TF) %>% select('orthogroup')))
)

tmp1 = names(table(tmp1))[table(tmp1) == 3]
tmp2 = names(table(tmp2))[table(tmp2) == 3]

Amniote_AST_TF <- rbind(Hsap_markers, Mmus_markers, Pvit_markers) %>% 
        filter(orthogroup %in% tmp1 & TF & cluster == 'Astrocytes')
Amniote_Oligo_TF <- rbind(Hsap_markers, Mmus_markers, Pvit_markers) %>% 
        filter(orthogroup %in% tmp2 & TF & cluster == 'Oligodendrocytes')

In [18]:
length(unique(c(unique(Amniote_AST_TF$orthogroup), unique(Amniote_Oligo_TF$orthogroup))))

[1] 14

In [19]:
write.table(rbind(Amniote_AST_TF, Amniote_Oligo_TF), file = 'Amniote_conserved.AST_vs_Oligo.TF_orthogroup.txt', 
            quote = F, sep = '\t', row.names= F, col.names = T)

In [22]:
Amniote_AST_TF %>% filter(species == 'Hsap') %>% arrange(orthogroup)
Amniote_AST_TF %>% filter(species == 'Mmus') %>% arrange(orthogroup)
Amniote_AST_TF %>% filter(species == 'Pvit') %>% arrange(orthogroup)

p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene,species,gene_name,TF,type,family,orthogroup
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
6.125171e-149,1.8222955,0.379,0.159,1.205434e-144,Astrocytes,ENSG00000181449,Hsap,SOX2,TRUE,WGD,"ENSG00000134595,ENSG00000182968,ENSG00000181449",OG0000314
0.000000e+00,2.3272859,0.856,0.306,0.000000e+00,Astrocytes,ENSG00000147862,Hsap,NFIB,TRUE,WGD,"ENSG00000147862,ENSG00000008441,ENSG00000141905,ENSG00000162599",OG0000557
0.000000e+00,1.8313343,0.969,0.694,0.000000e+00,Astrocytes,ENSG00000162599,Hsap,NFIA,TRUE,WGD,"ENSG00000147862,ENSG00000008441,ENSG00000141905,ENSG00000162599",OG0000557
2.690652e-278,1.1035126,0.761,0.426,5.295204e-274,Astrocytes,ENSG00000185630,Hsap,PBX1,TRUE,WGD,"ENSG00000204304,ENSG00000167081,ENSG00000185630,ENSG00000105717",OG0000672
0.000000e+00,7.4732953,0.298,0.002,0.000000e+00,Astrocytes,ENSG00000175745,Hsap,NR2F1,TRUE,WGD,"ENSG00000175745,ENSG00000160113,ENSG00000185551",OG0000810
0.000000e+00,8.9368446,0.348,0.001,0.000000e+00,Astrocytes,ENSG00000172201,Hsap,ID4,TRUE,Others,,OG0000815
0.000000e+00,8.8675270,0.331,0.002,0.000000e+00,Astrocytes,ENSG00000115738,Hsap,ID2,TRUE,WGD,"ENSG00000117318,ENSG00000125968,ENSG00000115738",OG0000815
2.550182e-261,4.6718004,0.247,0.015,5.018758e-257,Astrocytes,ENSG00000125398,Hsap,SOX9,TRUE,WGD,"ENSG00000100146,ENSG00000005513,ENSG00000125398",OG0000843
3.808893e-248,1.7263228,0.567,0.280,7.495901e-244,Astrocytes,ENSG00000134138,Hsap,MEIS2,TRUE,SSD,"ENSG00000105419,ENSG00000134138,ENSG00000143995",OG0000849


p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene,species,gene_name,TF,type,family,orthogroup
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
3.096900e-118,8.897776,0.103,0.000,5.818146e-114,Astrocytes,Sox1,Mmus,ENSMUSG00000096014,TRUE,WGD,"Sox3,Sox1,Sox2",OG0000314
1.128368e-232,3.396448,0.265,0.035,2.119865e-228,Astrocytes,Sox2,Mmus,ENSMUSG00000074637,TRUE,WGD,"Sox3,Sox1,Sox2",OG0000314
8.050312e-166,2.922109,0.234,0.047,1.512412e-161,Astrocytes,Nfia,Mmus,ENSMUSG00000028565,TRUE,WGD,"Nfia,Nfib,Nfic,Nfix",OG0000557
5.191682e-206,2.959807,0.291,0.065,9.753613e-202,Astrocytes,Nfib,Mmus,ENSMUSG00000008575,TRUE,WGD,"Nfia,Nfib,Nfic,Nfix",OG0000557
1.431817e-40,1.857554,0.124,0.051,2.689954e-36,Astrocytes,Nfic,Mmus,ENSMUSG00000055053,TRUE,WGD,"Nfia,Nfib,Nfic,Nfix",OG0000557
1.095303e-58,1.708834,0.189,0.084,2.057746e-54,Astrocytes,Nfix,Mmus,ENSMUSG00000001911,TRUE,WGD,"Nfia,Nfib,Nfic,Nfix",OG0000557
0.000000e+00,6.580214,0.297,0.007,0.000000e+00,Astrocytes,Pbx1,Mmus,ENSMUSG00000052534,TRUE,WGD,"Pbx3,Pbx2,Pbx1,Pbx4",OG0000672
3.790095e-225,7.651101,0.192,0.002,7.120451e-221,Astrocytes,Nr2f1,Mmus,ENSMUSG00000069171,TRUE,WGD,"Nr2f2,Nr2f1,Nr2f6",OG0000810
6.347003e-158,11.522532,0.134,0.000,1.192412e-153,Astrocytes,Id1,Mmus,ENSMUSG00000042745,TRUE,WGD,"Id1,Id3,Id2",OG0000815


p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene,species,gene_name,TF,type,family,orthogroup
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
3.130571e-184,3.1993767,0.216,0.027,5.913962e-180,Astrocytes,SOX1,Pvit,SOX1,TRUE,SSD,"SOX1,SOX14,SOX2,SOX21",OG0000314
1.031928e-47,1.1538497,0.171,0.075,1.949415e-43,Astrocytes,SOX2,Pvit,SOX2,TRUE,SSD,"SOX1,SOX14,SOX2,SOX21",OG0000314
1.789365e-36,0.6952689,0.339,0.230,3.380290e-32,Astrocytes,NFIA,Pvit,NFIA,TRUE,WGD,"NFIB,NFIC,NFIX,NFIA",OG0000557
4.878181e-85,0.6857972,0.700,0.558,9.215372e-81,Astrocytes,NFIX,Pvit,NFIX,TRUE,WGD,"NFIB,NFIC,NFIX,NFIA",OG0000557
8.327391e-22,0.6707825,0.206,0.134,1.573127e-17,Astrocytes,PBX1,Pvit,PBX1,TRUE,WGD,"PBX3,PBX1",OG0000672
1.556612e-239,2.7966951,0.320,0.065,2.940595e-235,Astrocytes,NR2F2,Pvit,NR2F2,TRUE,WGD,"NR2F2,NR2F6",OG0000810
0.000000e+00,4.0760691,0.858,0.131,0.000000e+00,Astrocytes,ID4,Pvit,ID4,TRUE,WGD,"ID4,ID3",OG0000815
0.000000e+00,3.4616739,0.490,0.052,0.000000e+00,Astrocytes,SOX9,Pvit,SOX9,TRUE,WGD,"SOX9,SOX10,SOX8",OG0000843
2.075540e-79,2.7319869,0.112,0.019,3.920903e-75,Astrocytes,MEIS2,Pvit,MEIS2,TRUE,Others,,OG0000849


In [23]:
Amniote_Oligo_TF %>% filter(species == 'Hsap') %>% arrange(orthogroup)
Amniote_Oligo_TF %>% filter(species == 'Mmus') %>% arrange(orthogroup)
Amniote_Oligo_TF %>% filter(species == 'Pvit') %>% arrange(orthogroup)

p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene,species,gene_name,TF,type,family,orthogroup
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
0.000000e+00,5.413560,0.784,0.046,0.000000e+00,Oligodendrocytes,ENSG00000146592,Hsap,CREB5,TRUE,WGD,"ENSG00000146592,ENSG00000170653,ENSG00000115966",OG0000608
2.740708e-131,6.293867,0.118,0.002,5.393713e-127,Oligodendrocytes,ENSG00000100146,Hsap,SOX10,TRUE,WGD,"ENSG00000100146,ENSG00000005513,ENSG00000125398",OG0000843
4.910426e-74,2.471779,0.110,0.020,9.663718e-70,Oligodendrocytes,ENSG00000005513,Hsap,SOX8,TRUE,WGD,"ENSG00000100146,ENSG00000005513,ENSG00000125398",OG0000843
0.000000e+00,2.710336,0.440,0.096,0.000000e+00,Oligodendrocytes,ENSG00000170802,Hsap,FOXN2,TRUE,WGD,"ENSG00000053254,ENSG00000170802",OG0001658
0.000000e+00,6.849033,0.543,0.008,0.000000e+00,Oligodendrocytes,ENSG00000148826,Hsap,NKX6-2,TRUE,SSD,"ENSG00000148826,ENSG00000163623,ENSG00000165066",OG0001688
0.000000e+00,2.831426,0.564,0.131,0.000000e+00,Oligodendrocytes,ENSG00000149557,Hsap,FEZ1,TRUE,SSD,"ENSG00000149557,ENSG00000171055",OG0002070
0.000000e+00,4.354516,0.323,0.022,0.000000e+00,Oligodendrocytes,ENSG00000184221,Hsap,OLIG1,TRUE,Others,,OG0013365


p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene,species,gene_name,TF,type,family,orthogroup
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
2.302492e-184,4.359110,0.176,0.008,4.325691e-180,Oligodendrocytes,Creb5,Mmus,ENSMUSG00000053007,TRUE,WGD,"Creb5,Gm28047,Atf2,Atf7",OG0000608
2.461712e-215,4.910077,0.190,0.004,4.624818e-211,Oligodendrocytes,Sox10,Mmus,ENSMUSG00000033006,TRUE,WGD,"Sox10,Sox8,Sox9",OG0000843
2.067617e-208,2.852038,0.239,0.028,3.884432e-204,Oligodendrocytes,Foxn3,Mmus,ENSMUSG00000033713,TRUE,WGD,"Foxn2,Foxn3",OG0001658
0.000000e+00,2.524181,0.413,0.036,0.000000e+00,Oligodendrocytes,Nkx6-2,Mmus,ENSMUSG00000041309,TRUE,SSD,"Nkx6-1,Nkx6-2,Nkx6-3",OG0001688
0.000000e+00,1.717791,0.711,0.286,0.000000e+00,Oligodendrocytes,Fez1,Mmus,ENSMUSG00000032118,TRUE,SSD,"Fez1,Fez2",OG0002070
0.000000e+00,2.149855,0.745,0.223,0.000000e+00,Oligodendrocytes,Olig1,Mmus,ENSMUSG00000046160,TRUE,Others,,OG0013365


p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene,species,gene_name,TF,type,family,orthogroup
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
4.512354e-183,3.536800,0.198,0.018,8.524288e-179,Oligodendrocytes,CREB5,Pvit,CREB5,TRUE,WGD,"ATF2,ATF7,CREB5",OG0000608
0.000000e+00,6.146551,0.502,0.009,0.000000e+00,Oligodendrocytes,SOX10,Pvit,SOX10,TRUE,WGD,"SOX9,SOX10,SOX8",OG0000843
4.058602e-146,2.409367,0.249,0.064,7.667105e-142,Oligodendrocytes,SOX8,Pvit,SOX8,TRUE,WGD,"SOX9,SOX10,SOX8",OG0000843
1.762072e-74,2.504373,0.116,0.023,3.328729e-70,Oligodendrocytes,FOXN2,Pvit,FOXN2,TRUE,WGD,"FOXN3,FOXN2",OG0001658
0.000000e+00,5.860532,0.264,0.005,0.000000e+00,Oligodendrocytes,NKX6-2,Pvit,NKX6-2,TRUE,WGD,"NKX6-1,NKX6-3,NKX6-2",OG0001688
0.000000e+00,1.525598,0.735,0.438,0.000000e+00,Oligodendrocytes,FEZ1,Pvit,FEZ1,TRUE,SSD,"FEZ1,FEZ2",OG0002070
2.182960e-231,2.375136,0.354,0.086,4.123830e-227,Oligodendrocytes,OLIG1,Pvit,OLIG1,TRUE,Others,,OG0013365


In [2]:
# for figure 2i 
fig2h_AST_Epen <- data.frame(species = rep(c('Hsap', 'Mmus', 'Pvit'), each = 4),
                             dup = rep(c('WGD', 'WGD', 'SSD', 'SSD'), times = 3),
                             type = rep(c('one', 'both', 'one', 'both'), times = 3),
                             number = c(642, 104, 512, 64, 395, 29, 326, 22, 408, 26, 211, 15))
fig2h_AST_Epen$species <- factor(fig2h_AST_Epen$species, levels = c('Hsap', 'Mmus', 'Pvit', 'Pmar'))
fig2h_AST_Epen$dup <- factor(fig2h_AST_Epen$dup, levels = c('WGD', 'SSD'))

pdf('Fig2i.AST_vs_Oligo.case_numbers.pdf', width = 4.5, height = 6)
fig2h_AST_Epen %>% ggbarplot(x = "type", y = "number", fill = "type", color = "type") + 
        scale_fill_manual(values=c("#C17F9E","#8BACD1"))+
        scale_color_manual(values=c("#C17F9E","#8BACD1"))+
        facet_grid(vars(dup), vars(species))
dev.off()

pdf 
  2